# Wav2Vec2 Binary Classification With Audio Augmentation

This notebook fine-tunes `facebook/wav2vec2-base-960h` for **binary PHQ classification** using participant-only audio.

**Task**
- Negative class: `PHQ < 10`
- Positive class: `PHQ >= 10`

**Pipeline**
1. Load participant-only WAV files from `processed/isolated_audio`.
2. Read labels and official splits from `processed/spectrograms/segment_metadata.xlsx`.
3. Segment each participant waveform into fixed windows and cache them on disk.
4. Apply **train-only waveform augmentation** inside the dataset.
5. Fine-tune a partially unfrozen Wav2Vec2 model.
6. Aggregate segment probabilities back to participant-level predictions.
7. Save checkpoints, metrics, predictions, and config for the best run.

**Why these augmentations first?**
- Mild gain perturbation helps reduce loudness dependence.
- Mild additive noise helps robustness to recording/background variation.
- Mild speed perturbation helps temporal invariance without distorting speech too aggressively.
- Small time shifts reduce sensitivity to segment boundaries.

Validation and test audio are kept untouched.


In [ ]:
# Run this only if you are on Colab after uploading the relevant processed files.
# !mkdir -p processed/spectrograms
# !mv segment_metadata.xlsx processed/spectrograms/
# !mv isolated_audio processed/

In [1]:
from __future__ import annotations


import json
import math
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn.functional as F
from scipy.signal import resample_poly
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoConfig,
    AutoProcessor,
    Wav2Vec2ForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# from google.colab import drive
# from pathlib import Path
# import shutil

# drive.mount('/content/drive')

# DRIVE_BASE = Path('/content/drive/MyDrive/a100_for_gpu')
# LOCAL_BASE = Path('/content/data')

# LOCAL_BASE.mkdir(parents=True, exist_ok=True)

# if not (LOCAL_BASE / 'processed').exists():
#     shutil.copytree(DRIVE_BASE / 'processed', LOCAL_BASE / 'processed')

# BASE_DIR = LOCAL_BASE


# BASE_DIR = Path('.')
BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / 'processed').exists() and (BASE_DIR.parent / 'processed').exists():
    BASE_DIR = BASE_DIR.parent
EXPERIMENTS_DIR = BASE_DIR / 'experiments'
ISOLATED_AUDIO_DIR = BASE_DIR / 'processed' / 'isolated_audio'
METADATA_XLSX = BASE_DIR / 'processed' / 'spectrograms' / 'segment_metadata.xlsx'
OUTPUT_ROOT = BASE_DIR / 'processed' / 'wav2vec_classification_aug'
MODEL_ROOT = EXPERIMENTS_DIR / 'best_model' / 'wav2vec_classification_aug'

HF_MODEL_NAME = 'facebook/wav2vec2-base-960h'
TARGET_SR = 16_000
SEGMENT_SEC = 8.0
HOP_SEC = 4.0
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
NUM_EPOCHS = 15
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.1
UNFREEZE_LAST_N_LAYERS = 2
EARLY_STOPPING_PATIENCE = 6
NUM_WORKERS = 0
RANDOM_STATE = 42
PHQ_THRESHOLD = 10
SPLITS = ('train', 'dev', 'test')

ENABLE_AUGMENTATION = True
AUG_GAIN_DB = 6.0
AUG_NOISE_PROB = 0.5
AUG_GAIN_PROB = 0.5
AUG_SPEED_PROB = 0.4
AUG_SHIFT_PROB = 0.5
AUG_MAX_SHIFT_SEC = 0.2
AUG_SPEED_RANGE = (0.95, 1.05)
AUG_NOISE_STD_RANGE = (0.001, 0.01)

DEVICE = torch.device(
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)

RUN_NAME = (
    f'wav2vec2_cls_seg{int(SEGMENT_SEC)}_hop{int(HOP_SEC)}'
    f'_u{UNFREEZE_LAST_N_LAYERS}_lr{LEARNING_RATE:g}'
    f'_aug{int(ENABLE_AUGMENTATION)}'
)
RUN_DIR = OUTPUT_ROOT / RUN_NAME
SEGMENT_CACHE_DIR = RUN_DIR / 'segment_cache'
BEST_MODEL_DIR = MODEL_ROOT / RUN_NAME

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
SEGMENT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

print(f'Audio dir      : {ISOLATED_AUDIO_DIR.resolve()}')
print(f'Metadata file  : {METADATA_XLSX.resolve()}')
print(f'Output root    : {OUTPUT_ROOT.resolve()}')
print(f'Run dir        : {RUN_DIR.resolve()}')
print(f'Model root     : {MODEL_ROOT.resolve()}')
print(f'Encoder        : {HF_MODEL_NAME}')
print(f'Device         : {DEVICE}')

if not ISOLATED_AUDIO_DIR.exists():
    raise FileNotFoundError(f'Isolated audio directory not found: {ISOLATED_AUDIO_DIR}')
if not METADATA_XLSX.exists():
    raise FileNotFoundError(f'Metadata spreadsheet not found: {METADATA_XLSX}')

Mounted at /content/drive
Audio dir      : /content/data/processed/isolated_audio
Metadata file  : /content/data/processed/spectrograms/segment_metadata.xlsx
Output root    : /content/data/processed/wav2vec_classification_aug
Run dir        : /content/data/processed/wav2vec_classification_aug/wav2vec2_cls_seg8_hop4_u2_lr1e-05_aug1
Model root     : /content/data/best_model/wav2vec_classification_aug
Encoder        : facebook/wav2vec2-base-960h
Device         : cuda


In [2]:
metadata = pd.read_excel(METADATA_XLSX)
for col in ['participant_id', 'phq_score', 'binary_label', 'participant_dur_s', 'n_turns']:
    metadata[col] = pd.to_numeric(metadata[col], errors='coerce')

metadata['participant_id'] = metadata['participant_id'].astype(int)
metadata['split'] = metadata['split'].astype(str).str.strip().str.lower()
metadata['target_label'] = (metadata['phq_score'] >= PHQ_THRESHOLD).astype(int)

participant_df = (
    metadata.sort_values(['participant_id', 'segment_idx'])
    .groupby('participant_id', as_index=False)
    .agg({
        'split': 'first',
        'phq_score': 'first',
        'binary_label': 'first',
        'target_label': 'first',
        'participant_dur_s': 'first',
        'n_turns': 'first',
    })
)

participant_df['session_name'] = participant_df['participant_id'].astype(str) + '_P'
participant_df['audio_path'] = participant_df['session_name'].map(lambda s: ISOLATED_AUDIO_DIR / f'{s}.wav')
participant_df['audio_exists'] = participant_df['audio_path'].map(Path.exists)

missing_audio = participant_df.loc[~participant_df['audio_exists'], ['participant_id', 'audio_path']]
if not missing_audio.empty:
    raise FileNotFoundError(
        'Missing isolated WAV files for some participants. Examples:\n' +
        missing_audio.head(10).to_string(index=False)
    )

display(participant_df.head())
print('Participants by split:')
print(participant_df['split'].value_counts().to_string())
print('\nClass balance by split:')
print(participant_df.groupby('split')['target_label'].value_counts().to_string())

,participant_id,split,phq_score,binary_label,target_label,participant_dur_s,n_turns,session_name,audio_path,audio_exists
0,300,test,2,0,0,155.76,87,300_P,/content/data/processed/isolated_audio/300_P.wav,True
1,301,test,3,0,0,475.44,104,301_P,/content/data/processed/isolated_audio/301_P.wav,True
2,302,dev,4,0,0,208.93,97,302_P,/content/data/processed/isolated_audio/302_P.wav,True
3,303,train,0,0,0,642.93,103,303_P,/content/data/processed/isolated_audio/303_P.wav,True
4,304,train,6,0,0,362.60,104,304_P,/content/data/processed/isolated_audio/304_P.wav,True


Participants by split:
split
train    107
test      47
dev       35

Class balance by split:
split  target_label
dev    0               23
       1               12
test   0               33
       1               14
train  0               76
       1               31


In [3]:
def load_audio_mono(path: Path, target_sr: int = TARGET_SR) -> np.ndarray:
    audio, sr = sf.read(path, always_2d=False)
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if sr != target_sr:
        audio = resample_poly(audio, target_sr, sr).astype(np.float32)
    return audio

def segment_audio(audio: np.ndarray, sample_rate: int, segment_sec: float, hop_sec: float) -> np.ndarray:
    segment_len = int(round(segment_sec * sample_rate))
    hop_len = int(round(hop_sec * sample_rate))
    if segment_len <= 0 or hop_len <= 0:
        raise ValueError('segment_sec and hop_sec must be positive')

    if len(audio) == 0:
        return np.zeros((1, segment_len), dtype=np.float32)

    segments = []
    start = 0
    while start + segment_len <= len(audio):
        segments.append(audio[start:start + segment_len])
        start += hop_len

    if not segments:
        padded = np.zeros(segment_len, dtype=np.float32)
        padded[:len(audio)] = audio[:segment_len]
        segments.append(padded)

    return np.stack(segments).astype(np.float32)

sample_audio = load_audio_mono(participant_df.iloc[0]['audio_path'])
sample_segments = segment_audio(sample_audio, TARGET_SR, SEGMENT_SEC, HOP_SEC)
print('Sample audio length (sec):', len(sample_audio) / TARGET_SR)
print('Segment batch shape      :', sample_segments.shape)

Sample audio length (sec): 155.76
Segment batch shape      : (37, 128000)


In [ ]:
segment_rows = []

for row in tqdm(participant_df.itertuples(index=False), total=len(participant_df), desc='Preparing segment cache'):
    cache_file = SEGMENT_CACHE_DIR / f'{int(row.participant_id)}.npz'

    if cache_file.exists():
        cached = np.load(cache_file)
        n_segments = int(cached['segments'].shape[0])
    else:
        audio = load_audio_mono(row.audio_path)
        segments = segment_audio(audio, TARGET_SR, SEGMENT_SEC, HOP_SEC)
        n_segments = int(segments.shape[0])
        np.savez_compressed(
            cache_file,
            participant_id=int(row.participant_id),
            duration_s=float(len(audio) / TARGET_SR),
            segments=segments.astype(np.float32),
        )

    for segment_idx in range(n_segments):
        segment_rows.append({
            'participant_id': int(row.participant_id),
            'session_name': str(row.session_name),
            'split': str(row.split),
            'phq_score': float(row.phq_score),
            'target_label': int(row.target_label),
            'participant_dur_s': float(row.participant_dur_s),
            'n_turns': int(row.n_turns),
            'cache_file': str(cache_file),
            'segment_idx': segment_idx,
            'n_segments': n_segments,
        })

segment_df = pd.DataFrame(segment_rows)
segment_df['participant_weight'] = 1.0 / segment_df['n_segments'].clip(lower=1)

display(segment_df.head())
print('Segments by split:')
print(segment_df['split'].value_counts().to_string())
print('Participants by split:')
print(participant_df['split'].value_counts().to_string())

Preparing segment cache:   0%|          | 0/189 [00:00<?, ?it/s]

## Train-Only Waveform Augmentation

Augmentation is only applied in the training dataset.
The implementation uses small perturbations to preserve label semantics while improving invariance.


In [ ]:
def random_gain(audio: np.ndarray, max_gain_db: float = AUG_GAIN_DB) -> np.ndarray:
    gain_db = np.random.uniform(-max_gain_db, max_gain_db)
    gain = float(10 ** (gain_db / 20.0))
    return np.clip(audio * gain, -1.0, 1.0).astype(np.float32)

def random_noise(audio: np.ndarray, std_range: tuple[float, float] = AUG_NOISE_STD_RANGE) -> np.ndarray:
    std = np.random.uniform(*std_range)
    noise = np.random.normal(0.0, std, size=audio.shape).astype(np.float32)
    return np.clip(audio + noise, -1.0, 1.0).astype(np.float32)

def random_time_shift(audio: np.ndarray, sample_rate: int = TARGET_SR, max_shift_sec: float = AUG_MAX_SHIFT_SEC) -> np.ndarray:
    max_shift = int(round(max_shift_sec * sample_rate))
    if max_shift <= 0:
        return audio
    shift = np.random.randint(-max_shift, max_shift + 1)
    if shift == 0:
        return audio
    shifted = np.zeros_like(audio)
    if shift > 0:
        shifted[shift:] = audio[:-shift]
    else:
        shifted[:shift] = audio[-shift:]
    return shifted.astype(np.float32)

def random_speed_perturb(audio: np.ndarray, speed_range: tuple[float, float] = AUG_SPEED_RANGE) -> np.ndarray:
    speed = np.random.uniform(*speed_range)
    if abs(speed - 1.0) < 1e-3:
        return audio

    up = 1000
    down = max(1, int(round(speed * up)))
    stretched = resample_poly(audio, up, down).astype(np.float32)

    if len(stretched) > len(audio):
        start = np.random.randint(0, len(stretched) - len(audio) + 1)
        stretched = stretched[start:start + len(audio)]
    elif len(stretched) < len(audio):
        padded = np.zeros_like(audio)
        padded[:len(stretched)] = stretched
        stretched = padded

    return np.clip(stretched, -1.0, 1.0).astype(np.float32)

def augment_waveform(audio: np.ndarray) -> np.ndarray:
    aug = audio.astype(np.float32, copy=True)
    if np.random.rand() < AUG_SHIFT_PROB:
        aug = random_time_shift(aug)
    if np.random.rand() < AUG_SPEED_PROB:
        aug = random_speed_perturb(aug)
    if np.random.rand() < AUG_GAIN_PROB:
        aug = random_gain(aug)
    if np.random.rand() < AUG_NOISE_PROB:
        aug = random_noise(aug)
    return np.clip(aug, -1.0, 1.0).astype(np.float32)

In [ ]:
processor = AutoProcessor.from_pretrained(HF_MODEL_NAME)

class SegmentDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, train_mode: bool = False):
        self.frame = frame.reset_index(drop=True).copy()
        self.train_mode = train_mode
        self._cache = {}

    def __len__(self) -> int:
        return len(self.frame)

    def _get_segment_bank(self, cache_file: str) -> np.ndarray:
        if cache_file not in self._cache:
            self._cache[cache_file] = np.load(cache_file)['segments'].astype(np.float32)
        return self._cache[cache_file]

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        segments = self._get_segment_bank(row.cache_file)
        audio = segments[int(row.segment_idx)].astype(np.float32, copy=True)

        if self.train_mode and ENABLE_AUGMENTATION:
            audio = augment_waveform(audio)

        return {
            'audio': audio,
            'label': float(row.target_label),
            'phq_score': float(row.phq_score),
            'participant_id': int(row.participant_id),
            'split': str(row.split),
            'segment_idx': int(row.segment_idx),
            'weight': float(row.participant_weight),
        }

class AudioClassificationCollator:
    def __init__(self, processor, sampling_rate: int = TARGET_SR):
        self.processor = processor
        self.sampling_rate = sampling_rate

    def __call__(self, batch: list[dict]) -> dict:
        inputs = self.processor(
            [item['audio'].tolist() for item in batch],
            sampling_rate=self.sampling_rate,
            return_tensors='pt',
            padding=True,
            return_attention_mask=True,
        )
        inputs['labels'] = torch.tensor([item['label'] for item in batch], dtype=torch.float32)
        inputs['weights'] = torch.tensor([item['weight'] for item in batch], dtype=torch.float32)
        inputs['participant_id'] = torch.tensor([item['participant_id'] for item in batch], dtype=torch.long)
        inputs['segment_idx'] = torch.tensor([item['segment_idx'] for item in batch], dtype=torch.long)
        inputs['phq_score'] = torch.tensor([item['phq_score'] for item in batch], dtype=torch.float32)
        return inputs

split_frames = {split: segment_df.loc[segment_df['split'] == split].reset_index(drop=True) for split in SPLITS}
datasets = {
    'train': SegmentDataset(split_frames['train'], train_mode=True),
    'dev': SegmentDataset(split_frames['dev'], train_mode=False),
    'test': SegmentDataset(split_frames['test'], train_mode=False),
}
collator = AudioClassificationCollator(processor)

train_loader = DataLoader(
    datasets['train'],
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collator,
)
eval_loaders = {
    split: DataLoader(
        datasets[split],
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collator,
    )
    for split in SPLITS
}

for split in SPLITS:
    label_counts = split_frames[split]['target_label'].value_counts().to_dict()
    print(
        f"{split:5s} | participants={participant_df[participant_df.split == split].shape[0]:3d} "
        f"| segments={split_frames[split].shape[0]:5d} | labels={label_counts}"
    )

In [ ]:
config = AutoConfig.from_pretrained(
    HF_MODEL_NAME,
    num_labels=1,
    problem_type='single_label_classification',
    final_dropout=0.1,
)

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    HF_MODEL_NAME,
    config=config,
    ignore_mismatched_sizes=True,
)

model.config.apply_spec_augment = False
if hasattr(model.wav2vec2, 'masked_spec_embed'):
    model.wav2vec2.masked_spec_embed = None

for param in model.parameters():
    param.requires_grad = False

for param in model.projector.parameters():
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True
for param in model.wav2vec2.feature_projection.parameters():
    param.requires_grad = True
for param in model.wav2vec2.encoder.layer_norm.parameters():
    param.requires_grad = True
for layer in model.wav2vec2.encoder.layers[-UNFREEZE_LAST_N_LAYERS:]:
    for param in layer.parameters():
        param.requires_grad = True

model.to(DEVICE)

total_params = sum(param.numel() for param in model.parameters())
trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
print(f'Trainable params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)')

In [ ]:
def move_batch_to_device(batch: dict, device: torch.device) -> dict:
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

def classification_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    y_pred = (y_prob >= threshold).astype(np.int64)

    metrics = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
    }

    if len(np.unique(y_true)) > 1:
        metrics['roc_auc'] = float(roc_auc_score(y_true, y_prob))
    else:
        metrics['roc_auc'] = float('nan')

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics.update({
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    })
    return metrics

def collect_segment_predictions(model, loader: DataLoader) -> pd.DataFrame:
    rows = []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader, total=len(loader), leave=False):
            batch = move_batch_to_device(batch, DEVICE)
            participant_id = batch.pop('participant_id').detach().cpu().numpy()
            segment_idx = batch.pop('segment_idx').detach().cpu().numpy()
            phq_score = batch.pop('phq_score').detach().cpu().numpy()
            weights = batch.pop('weights').detach().cpu().numpy()
            labels = batch['labels'].detach().cpu().numpy()

            outputs = model(
                input_values=batch['input_values'],
                attention_mask=batch['attention_mask'],
            )
            logits = outputs.logits.squeeze(-1)
            probs = torch.sigmoid(logits).detach().cpu().numpy()

            for i in range(len(participant_id)):
                rows.append({
                    'participant_id': int(participant_id[i]),
                    'segment_idx': int(segment_idx[i]),
                    'phq_score': float(phq_score[i]),
                    'target_label': int(labels[i]),
                    'segment_weight': float(weights[i]),
                    'prob_positive': float(probs[i]),
                })
    return pd.DataFrame(rows)

def aggregate_to_participants(segment_predictions: pd.DataFrame) -> pd.DataFrame:
    grouped = []
    for participant_id, frame in segment_predictions.groupby('participant_id', sort=True):
        weights = frame['segment_weight'].to_numpy(dtype=np.float32)
        if np.allclose(weights.sum(), 0.0):
            weights = np.ones_like(weights, dtype=np.float32)
        probs = frame['prob_positive'].to_numpy(dtype=np.float32)
        participant_prob = float(np.average(probs, weights=weights))
        grouped.append({
            'participant_id': int(participant_id),
            'phq_score': float(frame['phq_score'].iloc[0]),
            'target_label': int(frame['target_label'].iloc[0]),
            'predicted_probability': participant_prob,
            'predicted_label': int(participant_prob >= 0.5),
            'n_segments': int(len(frame)),
        })
    return pd.DataFrame(grouped)

def evaluate_participant_level(model, loader: DataLoader) -> tuple[pd.DataFrame, dict]:
    segment_predictions = collect_segment_predictions(model, loader)
    participant_predictions = aggregate_to_participants(segment_predictions)
    metrics = classification_metrics(
        participant_predictions['target_label'].to_numpy(dtype=np.int64),
        participant_predictions['predicted_probability'].to_numpy(dtype=np.float32),
    )
    return participant_predictions, metrics

In [ ]:
train_positive = int(participant_df.loc[participant_df['split'] == 'train', 'target_label'].sum())
train_total = int((participant_df['split'] == 'train').sum())
train_negative = train_total - train_positive
pos_weight_value = train_negative / max(train_positive, 1)
pos_weight = torch.tensor(pos_weight_value, dtype=torch.float32, device=DEVICE)
print(f'Positive class weight: {pos_weight_value:.4f}')

optimizer = AdamW(
    [param for param in model.parameters() if param.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_training_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

RESUME_FROM = None
SAVE_EVERY_N_STEPS = 200

start_epoch = 1
global_step = 0
best_dev_f1 = -float('inf')
history = []
epochs_without_improvement = 0
best_checkpoint = RUN_DIR / 'best_model.pt'
latest_checkpoint = RUN_DIR / 'checkpoint_latest.pt'

if RESUME_FROM is not None and Path(RESUME_FROM).exists():
    checkpoint = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    history = checkpoint.get('history', [])
    best_dev_f1 = checkpoint.get('best_dev_f1', -float('inf'))
    epochs_without_improvement = checkpoint.get('epochs_without_improvement', 0)
    global_step = checkpoint.get('global_step', 0)
    start_epoch = checkpoint['epoch'] + 1
    print(f"Resuming from epoch {checkpoint['epoch']}")

print('RUN_DIR:', RUN_DIR.resolve())
print('SEGMENT_CACHE_DIR:', SEGMENT_CACHE_DIR.resolve())

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0

    progress = tqdm(train_loader, total=len(train_loader), desc=f'Epoch {epoch}/{NUM_EPOCHS}')
    for step, batch in enumerate(progress, start=1):
        batch = move_batch_to_device(batch, DEVICE)
        batch.pop('participant_id')
        batch.pop('segment_idx')
        batch.pop('phq_score')
        weights = batch.pop('weights')

        outputs = model(
            input_values=batch['input_values'],
            attention_mask=batch['attention_mask'],
        )
        logits = outputs.logits.squeeze(-1)
        labels = batch['labels']

        per_item_loss = F.binary_cross_entropy_with_logits(
            logits,
            labels,
            reduction='none',
            pos_weight=pos_weight,
        )
        loss = (per_item_loss * weights).mean()

        if not torch.isfinite(loss):
            raise ValueError(f'Non-finite loss encountered at epoch={epoch}, step={step}: {loss.item()}')

        running_loss += float(loss.item())
        (loss / GRAD_ACCUM_STEPS).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            if global_step % SAVE_EVERY_N_STEPS == 0:
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'history': history,
                    'best_dev_f1': best_dev_f1,
                    'epochs_without_improvement': epochs_without_improvement,
                }, latest_checkpoint)

        progress.set_postfix(loss=running_loss / step)

    train_predictions, train_metrics = evaluate_participant_level(model, eval_loaders['train'])
    dev_predictions, dev_metrics = evaluate_participant_level(model, eval_loaders['dev'])

    epoch_row = {
        'epoch': epoch,
        'train_loss': running_loss / max(len(train_loader), 1),
        'train_accuracy': train_metrics['accuracy'],
        'train_balanced_accuracy': train_metrics['balanced_accuracy'],
        'train_f1': train_metrics['f1'],
        'train_roc_auc': train_metrics['roc_auc'],
        'dev_accuracy': dev_metrics['accuracy'],
        'dev_balanced_accuracy': dev_metrics['balanced_accuracy'],
        'dev_f1': dev_metrics['f1'],
        'dev_roc_auc': dev_metrics['roc_auc'],
    }
    history.append(epoch_row)

    state = {
        'epoch': epoch,
        'global_step': global_step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history,
        'best_dev_f1': best_dev_f1,
        'epochs_without_improvement': epochs_without_improvement,
    }

    torch.save(state, RUN_DIR / f'checkpoint_epoch_{epoch}.pt')
    torch.save(state, latest_checkpoint)
    pd.DataFrame(history).to_csv(RUN_DIR / 'history.csv', index=False)

    print(json.dumps(epoch_row, indent=2))
    print(f"Saved epoch checkpoint to: {(RUN_DIR / f'checkpoint_epoch_{epoch}.pt').resolve()}")

    if dev_metrics['f1'] > best_dev_f1:
        best_dev_f1 = dev_metrics['f1']
        epochs_without_improvement = 0
        state['best_dev_f1'] = best_dev_f1
        torch.save(state, best_checkpoint)
        torch.save(state, BEST_MODEL_DIR / 'best_model.pt')
        with (BEST_MODEL_DIR / 'config.json').open('w', encoding='utf-8') as f:
            json.dump({
                'run_name': RUN_NAME,
                'hf_model_name': HF_MODEL_NAME,
                'segment_sec': SEGMENT_SEC,
                'hop_sec': HOP_SEC,
                'num_epochs': NUM_EPOCHS,
                'learning_rate': LEARNING_RATE,
                'unfreeze_last_n_layers': UNFREEZE_LAST_N_LAYERS,
                'threshold': PHQ_THRESHOLD,
                'augmentation_enabled': ENABLE_AUGMENTATION,
            }, f, indent=2)
        print(f'Saved best checkpoint to: {best_checkpoint.resolve()}')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print('Early stopping triggered.')
            break

In [ ]:
history_df = pd.DataFrame(history)
display(history_df)

checkpoint = torch.load(best_checkpoint, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE)
model.eval()

final_metrics = []
for split in SPLITS:
    predictions, metrics = evaluate_participant_level(model, eval_loaders[split])
    predictions.to_csv(RUN_DIR / f'{split}_participant_predictions.csv', index=False)
    final_metrics.append({'split': split, **metrics})
    print(
        f"{split:5s} | acc={metrics['accuracy']:.4f} | bal_acc={metrics['balanced_accuracy']:.4f} "
        f"| f1={metrics['f1']:.4f} | auc={metrics['roc_auc']:.4f}"
    )

final_metrics_df = pd.DataFrame(final_metrics)
history_df.to_csv(RUN_DIR / 'history.csv', index=False)
final_metrics_df.to_csv(RUN_DIR / 'participant_metrics.csv', index=False)
participant_df.to_csv(RUN_DIR / 'participant_index.csv', index=False)

display(final_metrics_df)
print(f'Saved run artifacts to: {RUN_DIR.resolve()}')

final_metrics_df.to_csv(BEST_MODEL_DIR / 'participant_metrics.csv', index=False)
for split in SPLITS:
    src = RUN_DIR / f'{split}_participant_predictions.csv'
    dst = BEST_MODEL_DIR / f'{split}_participant_predictions.csv'
    if src.exists():
        dst.write_bytes(src.read_bytes())
print(f'Saved best-model artifacts to: {BEST_MODEL_DIR.resolve()}')

## Notes

- This notebook uses **custom BCE-with-logits loss**, so it does not rely on the Hugging Face model's built-in classification loss.
- The primary early-stopping metric is **participant-level dev F1**.
- If the model becomes unstable, first try lowering `LEARNING_RATE` to `5e-6` or disabling one augmentation at a time.
- If recall is poor on the positive class, consider increasing overlap or tuning the decision threshold on the dev set after training.
